# Module 12 Lab - Ethics, Fairness, and Bias in ML

**Objective:** To understand how machine learning models can inherit and amplify societal biases, how to measure this bias using fairness metrics, and to think critically about the ethical implications of deploying ML systems.

**In this lab, you will train a model on a real-world dataset and audit it for fairness across different demographic groups.**

## Part 1: What is Algorithmic Bias?

**Concept:** Machine learning models learn from data. If the data reflects existing societal biases, the model will learn those biases. An "unbiased" algorithm trained on biased data will produce a biased model. This can lead to systems that are systematically unfair to certain groups of people.

**Sources of Bias:**
*   **Historical Bias:** The data reflects a world with historical injustices (e.g., past hiring data may show fewer women in leadership roles).
*   **Measurement Bias:** The way we collect or measure data is flawed (e.g., using arrest records as a proxy for crime, which can be influenced by policing patterns).
*   **Representation Bias:** The data underrepresents certain groups, so the model doesn't learn to perform well for them.

**Problem:** We will use the "Adult" dataset, which is used to predict whether an individual's income is greater than $50k/year. It contains sensitive attributes like `sex` and `race`, which we can use to audit our model for bias.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

# Load the data
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']
df = pd.read_csv(url, header=None, names=columns, sep=r',\s*', engine='python', na_values='?')

# Data Cleaning
df.dropna(inplace=True)
df['income'] = df['income'].map({'<=50K': 0, '>50K': 1})

X = df.drop('income', axis=1)
y = df['income']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Create a preprocessing pipeline
numeric_features = X.select_dtypes(include='number').columns
categorical_features = X.select_dtypes(exclude='number').columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown='ignore'), categorical_features)
)

# Train a baseline model
model = make_pipeline(preprocessor, LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

print(f"Overall model accuracy: {model.score(X_test, y_test):.2%}")

Overall model accuracy: 84.61%


## Part 2: Auditing the Model for Fairness

High overall accuracy can hide poor performance on specific subgroups. We need to audit the model by comparing its performance across sensitive attributes like `sex`.

**Concept: Group Fairness**
One common fairness goal is to ensure the model works equally well for different groups. We can measure this by calculating metrics for each group separately.

**Your Task:** Create a function to calculate accuracy for different subgroups and then use it to compare the model's performance for males and females.

In [ ]:
# --- ENTER YOUR CODE HERE ---

def get_subgroup_accuracy(model, X_test, y_test, subgroup_column, subgroup_value):
    """Calculates accuracy for a specific subgroup of the test data."""
    # 1. Create a boolean mask to select the subgroup from X_test
    subgroup_mask = X_test[subgroup_column] == subgroup_value

    # 2. Select the subgroup data
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    # 3. Calculate and return the model's score on this subgroup
    return model.score(X_subgroup, y_subgroup)

# 4. Calculate accuracy for males and females
acc_male = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Male')
acc_female = get_subgroup_accuracy(model, X_test, y_test, 'sex', 'Female')

print(f"Accuracy for Males: {acc_male:.2%}")
print(f"Accuracy for Females: {acc_female:.2%}")

Accuracy for Males: 81.20%
Accuracy for Females: 91.81%


### Task 2: Deeper Dive with a Confusion Matrix

Accuracy alone doesn't tell the whole story. Let's look at the types of errors the model makes for each group.

**Your Task:** Calculate and compare the **False Positive Rate (FPR)** and **False Negative Rate (FNR)** for males and females.

*   **FPR:** `FP / (FP + TN)` - The percentage of people who did NOT have high income but were incorrectly predicted to have high income.
*   **FNR:** `FN / (FN + TP)` - The percentage of people who DID have high income but were incorrectly predicted to have low income.

In [ ]:
from sklearn.metrics import confusion_matrix

def get_rates(model, X_test, y_test, subgroup_column, subgroup_value):
    subgroup_mask = X_test[subgroup_column] == subgroup_value
    X_subgroup = X_test[subgroup_mask]
    y_subgroup = y_test[subgroup_mask]

    y_pred_subgroup = model.predict(X_subgroup)
    tn, fp, fn, tp = confusion_matrix(y_subgroup, y_pred_subgroup).ravel()

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)
    return fpr, fnr

# --- ENTER YOUR CODE HERE ---

# 1. Calculate the rates for males and females
fpr_male, fnr_male = get_rates(model, X_test, y_test, 'sex', 'Male')
fpr_female, fnr_female = get_rates(model, X_test, y_test, 'sex', 'Female')

print(f"Male - False Positive Rate: {fpr_male:.2%}, False Negative Rate: {fnr_male:.2%}")
print(f"Female - False Positive Rate: {fpr_female:.2%}, False Negative Rate: {fnr_female:.2%}")

Male - False Positive Rate: 10.26%, False Negative Rate: 37.80%
Female - False Positive Rate: 2.81%, False Negative Rate: 47.84%


## 📝 Reflective Knowledge Check

**Instructions:** Answer the following questions in this markdown cell. Your answers should be based on **your specific results** from the code you ran above.

### **1. Analyze Your Results:** Look at the subgroup accuracies you calculated. Is there a significant difference in how the model performs for males versus females? Which group does the model perform better for?

**[Huong]:** Yes. The model performs much better for females based on raw accuracy. Female accuracy is **91.81%**, while male accuracy is **81.20%**, a gap of about **10.61 percentage points**. That is large enough to raise a fairness concern because it shows the model is not performing equally across groups, even though the overall accuracy is fairly strong at **84.61%**.

**[Alissa]:**

Yes, there is a clear and meaningful difference in performance between the two groups. The model achieves **91.81% accuracy for females** and **81.20% for males**, resulting in a gap of approximately **10.61 percentage points**.

This indicates that the model performs significantly better for females. While the overall accuracy of **84.61%** might suggest the model is reliable at a high level, this average masks an important imbalance between subgroups.

From an AI perspective, this is a fairness concern because the model is not providing consistent performance across different populations. In real-world applications, such disparities can lead to unequal outcomes, where one group benefits from more accurate predictions while another is more likely to experience errors.

**[Stuart]:**

The model shows a 10% better accuracy for female over male accuracy (91.81% to 81.2%). The model's overall accuracy is 84.61%. The model's accuracy difference is significant across the groups.

**[Ruben]: [ENTER YOUR ANSWERS HERE]**

### **2. Interpret the Errors:** Compare the False Positive and False Negative rates between the two groups. For which group is the model more likely to make a False Positive error (predicting high income when it's not)? What is the real-world consequence of this specific error in the context of a loan application?

**[Huong]:** The model is much more likely to make a **False Positive** error for **males**. The male **FPR is 10.26%**, compared with only **2.81%** for females. That means males are more often incorrectly predicted as having income above $50K when they actually do not. In a loan application setting, this error could result in approving loans for applicants whose income does not support repayment, which increases lender risk and can also place applicants into financial stress if they receive loans they cannot realistically afford.

**[Alissa]:**

The model is more likely to make a **False Positive error for males**. The **False Positive Rate (FPR) for males is 10.26%**, compared to only **2.81% for females**, which shows a substantial difference between the groups.

This means the model more frequently predicts that male applicants earn above $50K when they actually do not. In the context of a loan application, this type of error is particularly risky because it can lead to approving loans for individuals who may not have the financial capacity to repay them.

From a real-world perspective, this increases risk for both sides. Lenders face higher default rates, while applicants may be pushed into financial situations they cannot sustain. Over time, this kind of imbalance can also indicate a systemic issue in the model, where one group is more exposed to risky approvals due to inaccurate predictions.

**[Stuart]:**

The male false positive rate is 10.26% where the female false positive rate is 2.81%. The male false negative rate is 37.8% where the female false negative rate is 47.84%. 
Males are more likely to indicate for a false positive, and therefore have a greater chance of being approved for a loan that should have been denied, creating a greater risk of default on the debt.

**[Ruben]: [ENTER YOUR ANSWERS HERE]**

### **3. Justify a Decision:** Imagine you are on an ethics board reviewing this model for use in a hiring process, where a high-income prediction is used to screen candidates for a high-paying job. Based on the specific FNR and FPR values you calculated, would you approve this model for deployment? Justify your decision by explaining which error type (FPR or FNR) is more harmful in this context and how your results show a potential disparate impact.

**[Huong]:** I would **not approve** this model for deployment in a hiring screen. In this context, the more harmful error is the **False Negative**, because it means a qualified person is screened out from consideration for a high-paying opportunity. The results show the **female FNR is 47.84%**, which is much worse than the **male FNR of 37.80%**. That means women who actually belong in the high-income class are being rejected at a substantially higher rate. This creates a clear risk of **disparate impact** because one group is more likely to be denied access to opportunity even when they meet the target outcome.

**[Alissa]:**

I would **not approve** this model for a hiring process.

In this case, the most harmful error is the False Negative, because it means qualified candidates are being filtered out too early. The model shows a higher **FNR for females (47.84%)** compared to **males (37.80%)**, which means women are more likely to be incorrectly rejected even when they meet the criteria.

This creates a clear **disparate impact**, since one group is being denied opportunities at a higher rate. Before using this model, it would need adjustments to make the outcomes more balanced.

**[Stuart]:**

This model should not be approved for a hiring process as it would disproportionately deny women (47.84% FNR) compared to men (37.8% FNR).  The FNR is also quite high overall and has the potential to deny 1/3 to 1/2 of candidates improperly.

**[Ruben]: [ENTER YOUR ANSWERS HERE]**

### **4. Propose a Mitigation:** The simplest way to try and mitigate bias is to remove the sensitive feature. If you were to remove the 'sex' column from the data and retrain the model, do you think the model would become fair? Why or why not? (Hint: Think about what other columns might be correlated with 'sex').

**[Huong]:** No, removing the **'sex'** column alone would probably **not make the model fair**. Other features in the Adult dataset can still act as proxies for sex, such as **occupation**, **relationship**, **marital-status**, **hours-per-week**, and even patterns in **education** or **capital-gain**. Because those features may still carry gender-related structure, the model can continue to reproduce biased outcomes even without directly seeing the sex column. A better mitigation approach would include fairness auditing, feature review, threshold analysis, and possibly fairness-aware model constraints rather than only dropping one sensitive attribute.

**[Alissa]:**

No, removing the **'sex'** column alone would likely not make the model fair.

Other features can still act as proxies, like **occupation, marital status**, or **hours worked**, which may indirectly reflect gender patterns. Because of that, the model can continue learning biased behavior even without explicitly using the sex variable.

To actually improve fairness, it would be better to combine this with other steps, like reviewing features, checking bias metrics, and adjusting the model rather than just removing one column.

**[Stuart]:**

Removing the sex column as a mitigation would not remove the model's bias, it would simply remove the transparency of the problem. "Occupation", "marital-status", and "education" could all be similar markers to point to sex as a probability, and could affect the model in a similar way. Outside the scope of this lab, application of tools such as aif360 to reweigh models that have high Disparate Impact or Average Odds Difference can produce a model that mitigates bias in the model's performance.

**[Ruben]: [ENTER YOUR ANSWERS HERE]**